# WABAD paper example, step by step

Follow the real `ARD_20211027_072000.wav` recording used in the paper examples. Every section explains **Input → Processing → Output**. Obtain the matching waveform and expert annotation table from the author’s study Zenodo release before running the analysis; see [data release instructions](../docs/DATA_RELEASE.md). The release URL and DOI are pending in `data_release.json`.

**Input:** the actual WABAD recording and reference table. **Processing:** fine-band drift grouping and per-group GMM segmentation. **Output:** candidate time–frequency boxes, not species names or separated voices. The expert labels are used for reference plots and evaluation, not training.

The author’s original figures are included unchanged for inspection. No audio or annotation coordinates have been reconstructed from them.

![Original ARD expert-reference spectrogram and species legend](../examples/paper/ARD_20211027_072000/ground_truth.png)

Install from the README, select the installed environment’s Python kernel, and run cells from top to bottom.

## 1. Select the real-data configuration — check this cell

**Input:** the repository and `configs/paper_example.json`. **Processing:** locate the repository and resolve the input/output paths. **Output:** complete settings for the ARD paper example.

Check `audio_file` and `annotation_file` in the JSON against your downloaded study files. Paths in JSON are relative to the JSON file. For a different WABAD recording, change those paths explicitly. The notebook does not generate or substitute input data.

In [ ]:
from pathlib import Path
from adaptive_windowing.config import load_config

# CHECK THIS: the default config selects the ARD recording from the paper.
CONFIG_NAME = "paper_example.json"

REPO = Path.cwd()
if not (REPO / "pyproject.toml").exists():
    REPO = REPO.parent  # Jupyter may start inside notebooks/.
if not (REPO / "pyproject.toml").exists():
    raise FileNotFoundError("Start Jupyter from the repository or notebooks folder.")
CONFIG_PATH = REPO / "configs" / CONFIG_NAME
cfg = load_config(CONFIG_PATH)
print("Audio:", cfg.get("audio_file") or cfg.get("data_root"))
print("Expert references:", cfg.get("annotation_file"))
print("Results:", cfg["output_dir"])

## 2. Check the downloaded study files

**Input:** the real recording and matching reference table selected by the config. **Processing:** confirm both are installed, then validate audio duration, column names, time and frequency units. **Output:** a recording record and expert boxes.

The default expects `data/WABAD/ARD/recordings/ARD_20211027_072000.wav` and its annotation table. If the final Zenodo archive uses another annotation suffix or folder, update `annotation_file` in the config. Missing input files stop the workflow with instructions; another recording is not silently substituted.

In [ ]:
from adaptive_windowing.pipeline import records_for, validate_record
from adaptive_windowing import research as method

required = [Path(cfg[key]) for key in ("audio_file", "annotation_file") if cfg.get(key)]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Install the real WABAD study files from the author's Zenodo release first. "
        "See docs/DATA_RELEASE.md and check configs/paper_example.json. Missing: "
        + "; ".join(missing)
    )
records = records_for(cfg)
RECORDING_INDEX = 0  # Zero selects the explicit ARD file in the default config.
record = records[RECORDING_INDEX]
references, info = validate_record(record, cfg["parameters"])
method.configure(cfg["parameters"])
p = cfg["parameters"]
print(record)
print(f"{info.duration:.2f} seconds; {info.samplerate} Hz; {info.channels} channel(s)")
print("Expert references:", "not supplied" if references is None else len(references))

## 3. Prepare audio and describe short moments

**Input:** one audio file. **Processing:** mono conversion, resampling, optional HPSS preprocessing, overlapping micro-windows, Mel power spectra and PCEN per micro-window. **Output:** a waveform, window times and a spectral matrix shaped `(number of windows, number of Mel bins)`.

Defaults: 22,050 Hz; 0.25-s windows; 0.125-s steps; 128 Mel bins. Fixed micro-windows are measurements; the final events can have different durations.

In [ ]:
import librosa
import numpy as np

y, sr = librosa.load(str(record["audio"]), sr=p["TARGET_SR"], mono=True)
y = method.apply_waveform_preprocessing(y, sr)
starts, ends = method.make_fixed_windows(len(y) / sr, p["WINDOW_S"], p["HOP_S"])
vectors = method.compute_window_repr_vectors(
    y, sr, starts, ends, p["N_MELS_FULL"], p["FMIN"], p["FMAX"], use_pcen=p["USE_PCEN"]
)
if p["DENOISE_METHOD"] == "median_specsub":
    vectors = method.subtract_stationary_band_floor(
        vectors, strength=p["SPEC_FLOOR_STRENGTH"], floor_mode=p["SPEC_FLOOR_MODE"]
    )
print("Spectral matrix:", vectors.shape)

## 4. Measure acoustic change in fine bands

**Input:** the spectral matrix. **Processing:** split it into 20 Mel-spaced bands, measure Euclidean change between adjacent windows, smooth the curves, and median/MAD-standardise them. **Output:** one drift curve per band, plus peak evidence used by the alternative Jaccard similarity.

A high drift value means that the local spectrum changed. It does not establish which species or source caused the change.

In [ ]:
edges = method.band_edges_hz(p["FMIN"], p["FMAX"], p["N_FINE_BANDS"], p["BAND_SCALE"])
mel_freqs = librosa.mel_frequencies(n_mels=p["N_MELS_FULL"], fmin=p["FMIN"], fmax=p["FMAX"])
band_vectors = method.split_mel_vectors_into_bands(vectors, mel_freqs, edges)
band_drifts = method.compute_direct_band_drifts(band_vectors, p["DRIFT_SMOOTH_K"])
standardised, evidence = [], []
for curve in band_drifts:
    z, binary, _, _ = method.detect_peaks_from_drift(curve, p["PEAK_K_MAD"], p["PEAK_MIN_SEP_WINDOWS"])
    standardised.append(z)
    evidence.append(binary)
print("Drift matrix:", np.asarray(standardised).shape)

## 5. Group bands with related behaviour

**Input:** drift curves, within-band profile descriptors, and frequency order. **Processing:** combine temporal coherence, descriptor similarity and adjacency, then use average-linkage hierarchical clustering. **Output:** group labels and a median drift curve per group.

The defaults weight the three cues at 0.70, 0.20 and 0.10. Groups describe local acoustic behaviour; their numbers are not species IDs.

Compare the computed group trajectories with the author's supplied original figure below. Group IDs and plotting details can differ if the run settings or dependencies differ; the original complete run manifest was not supplied.

![Original grouped drift trajectories for the ARD recording](../examples/paper/ARD_20211027_072000/cluster_drifts.png)

In [ ]:
descriptors = method.compute_band_frequency_descriptors(
    band_vectors, k_mad_activity=p["FREQ_DESC_ACTIVITY_K_MAD"],
    normalize_subband=p["FREQ_DESC_NORMALIZE_SUBBAND"], use_activity=p["FREQ_DESC_USE_ACTIVITY"]
)
s_time = method.build_time_similarity_matrix(standardised, evidence, p["SIMILARITY_MODE"], p["MAX_LAG_WINDOWS"])
s_profile = method.build_frequency_similarity_matrix(descriptors)
s_adjacency = method.build_adjacency_similarity_matrix(p["N_FINE_BANDS"], p["ADJ_TAU"])
affinity = method.combine_similarity_matrices(s_time, s_profile, s_adjacency, p["ALPHA_TIME"], p["BETA_FREQ"], p["GAMMA_ADJ"])
_, group_labels = method.cluster_bands_from_similarity(
    affinity, p["LINKAGE_METHOD"], p["N_GROUPS"], p["DISTANCE_THRESHOLD"]
)
group_ids, group_drifts, members = method.aggregate_cluster_drifts(standardised, group_labels, p["CLUSTER_AGG"])
print("Groups:", len(group_ids), "| member band indices:", members)

## 6. Turn group drift into event boxes

**Input:** group drift trajectories, window times, and frequency edges. **Processing:** fit two Gaussian states per group, choose the higher-mean state, apply the posterior threshold, smooth labels, merge short gaps and remove short runs. **Output:** event boxes and GMM diagnostics.

No references are passed to this cell. If you change detection settings, edit the config and rerun from the top. A posterior threshold of 0.50 is the supplied default; it is not a species confidence.

In [ ]:
boxes, states = method.gmm_boxes_from_cluster_drifts(
    group_drifts, members, group_ids, edges, starts, ends,
    min_event_posterior=p["GMM_MIN_EVENT_POSTERIOR"],
    smooth_labels_k=p["GMM_SMOOTH_LABELS_K"],
    merge_gap_windows=p["GMM_MERGE_GAP_WINDOWS"],
    min_event_windows=p["GMM_MIN_EVENT_WINDOWS"],
    min_event_area_q=p["GMM_MIN_EVENT_AREA_Q"],
    use_log_drift=p["GMM_USE_LOG_DRIFT"], random_state=p["GMM_RANDOM_STATE"]
)
import pandas as pd
print("Candidate events:", len(boxes))
pd.DataFrame(boxes).head()

## 7. Save and inspect this example

**Input:** the candidate boxes, prepared audio and optional references. **Processing:** write a coordinate table and draw candidate/reference overlays. **Output:** a CSV and PNG in `notebook_preview`.

Solid cyan means a candidate event; dashed green means a reference box. The plot shows preprocessed/resampled audio, as in the supplied prediction code.

The original paper-example prediction image is shown below for reference. It contains cyan proposals only; the new preview above/below also includes dashed green expert boxes when the annotation table is supplied.

![Original ARD prediction boxes in cyan](../examples/paper/ARD_20211027_072000/predicted_boxes.png)

In [ ]:
from adaptive_windowing.io import save_boxes
from IPython.display import display, Image
preview_dir = Path(cfg["output_dir"]) / "notebook_preview" / record["audio"].stem
preview_dir.mkdir(parents=True, exist_ok=True)
save_boxes(boxes, preview_dir / "predicted_boxes.csv")
method.save_spectrogram_with_boxes(
    y, sr, boxes, references, preview_dir / "spectrogram_boxes.png",
    title="Candidate events (cyan); references (green dashed)"
)
display(Image(filename=str(preview_dir / "spectrogram_boxes.png")))

## 8. Optional reference-based evaluation

**Input:** saved candidate geometry and reference geometry. **Processing:** apply the new manuscript-oriented conditional criteria. **Output:** metrics as fractions, not percentages.

This evaluator was reconstructed from manuscript §4.2. It has not been checked against the paper's full original analysis. See `docs/EVALUATION.md` for matching choices, conditional filtering and the separate notebook metrics. Missing references give no accuracy estimate.

In [ ]:
from adaptive_windowing.evaluation import conditional_metrics
if references is None:
    print("No references supplied: inspect proposals manually.")
else:
    display(pd.Series(conditional_metrics(boxes, references)))

## 9. Run and save the standard workflow

**Input:** the same ARD configuration, or the full study config when explicitly selected. **Processing:** repeat detection with validation, evaluation, plots, completion checks and an environment manifest. **Output:** the standard per-recording files and summary.

This cell intentionally repeats the earlier teaching steps, unless matching completed results already exist. Keep `RUN_FULL_STUDY = False` for the ARD paper example. Set it to `True` only after configuring `configs/wabad.json` for the full downloaded study dataset.

In [ ]:
from adaptive_windowing.pipeline import run

RUN_FULL_STUDY = False  # CHANGE THIS only when ready to run the full study dataset.
run_config = REPO / "configs/wabad.json" if RUN_FULL_STUDY else CONFIG_PATH
summary = run(run_config)
summary.head()